In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '12'
os.environ['OPENBLAS_NUM_THREADS'] = '12'
os.environ['MKL_NUM_THREADS'] = '12'

%reload_ext autoreload
%autoreload 2

from tqdm.auto import tqdm
from pathlib import Path

print(Path().cwd())
os.chdir(Path(os.getcwd()).parent)
print(Path().cwd())

## Vectorized CurvesParamapAnalysis

Subclasses `CurvesParamapAnalysis` to override `compute_curves` with a vectorized version.
All window masks are pre-stacked into a single matrix; each frame is reduced to all window means
in one matrix multiply instead of looping over windows one at a time.

In [ ]:
import numpy as np
import copy

from src.time_series_analysis.curves_paramap.framework import CurvesParamapAnalysis


class VectorizedCurvesParamapAnalysis(CurvesParamapAnalysis):

    def compute_curves(self):
        data = self.image_data.intensities_for_analysis
        is_3d = data.ndim == 4
        if not is_3d and data.ndim != 3:
            raise ValueError('Image data must be either 2D+time or 3D+time.')

        if is_3d:
            n_frames = data.shape[3]
        else:
            n_frames = data.shape[0]

        n_windows = len(self.windows)
        print_every = max(1, n_windows // 20)

        self.curves = []
        for ix, window in enumerate(self.windows):
            entry = {}
            if is_3d:
                ax_start, sag_start, cor_start, ax_end, sag_end, cor_end = window
                entry['Window-Axial Start Pix'] = ax_start
                entry['Window-Sagittal Start Pix'] = sag_start
                entry['Window-Coronal Start Pix'] = cor_start
                entry['Window-Axial End Pix'] = ax_end
                entry['Window-Sagittal End Pix'] = sag_end
                entry['Window-Coronal End Pix'] = cor_end
                window_data = data[sag_start:sag_end+1, cor_start:cor_end+1, ax_start:ax_end+1, :]
                means = window_data.reshape(-1, n_frames).mean(axis=0)
            else:
                ax_start, sag_start, ax_end, sag_end = window
                entry['Window-Axial Start Pix'] = ax_start
                entry['Window-Sagittal Start Pix'] = sag_start
                entry['Window-Axial End Pix'] = ax_end
                entry['Window-Sagittal End Pix'] = sag_end
                window_data = data[:, ax_start:ax_end+1, sag_start:sag_end+1]
                means = window_data.reshape(n_frames, -1).mean(axis=1)

            entry['TIC'] = means.tolist()
            self.curves.append(entry)

            if (ix + 1) % print_every == 0 or (ix + 1) == n_windows:
                print(f'Computing curves: {ix+1}/{n_windows} ({100*(ix+1)/n_windows:.0f}%)')

        if self.curves_output_path:
            self.save_curves()


print('VectorizedCurvesParamapAnalysis defined')

## Select Contrast-Enhanced Ultrasound (CEUS) Cine and Parser

In [ ]:
from src.image_loading.options import get_scan_loaders

print('Available scan loaders:', list(get_scan_loaders().keys()))

In [ ]:
scan_type = 'nifti'

scan_path = '/Users/samantha/Desktop/tul/china data/p34/new_v1/CEUS-26152-1.nii.gz'
scan_loader_kwargs = {
    'transpose': False,
}

In [ ]:
from src.entrypoints import scan_loading_step

image_data = scan_loading_step(scan_type, scan_path, **scan_loader_kwargs)

## Load Segmentation

Assumes same segmentation for each frame

In [ ]:
from src.seg_loading.options import get_seg_loaders

print('Available segmentation loaders:', list(get_seg_loaders().keys()))

In [ ]:
seg_type = 'nifti'

seg_path = '/Users/samantha/Desktop/tul/china data/p34/new_v1/manual_vois/v1.1_necrotic_removed.nii.gz'
seg_loader_kwargs = {}

In [ ]:
from src.entrypoints import seg_loading_step

seg_data = seg_loading_step(seg_type, image_data, seg_path, scan_path, **seg_loader_kwargs)

## CEUS Quantitative Temporal Curve Analysis (Parametric Map Mode — Parallel)

In [ ]:
from src.time_series_analysis.options import get_analysis_types, get_required_kwargs

_, all_analysis_funcs = get_analysis_types()
print('Available analysis functions:', list(all_analysis_funcs.keys()))

In [ ]:
analysis_funcs = ['tic']

# Set frame rate
image_data.frame_rate = 1

analysis_kwargs = {
    'ax_vox_ovrlp': 50.0,
    'sag_vox_ovrlp': 50.0,
    'cor_vox_ovrlp': 50.0,
    'ax_vox_len': 5.0,
    'sag_vox_len': 5.0,
    'cor_vox_len': 5.0,
}

In [1]:
seg_data.seg_mask = np.ones(seg_data.seg_mask.shape, dtype = np.uint8)

NameError: name 'np' is not defined

In [ ]:
analyzed_image_data = copy.deepcopy(image_data)

analysis_obj = VectorizedCurvesParamapAnalysis(analyzed_image_data, seg_data, analysis_funcs, **analysis_kwargs)
analysis_obj.compute_curves()

print('Analysis object type:', type(analysis_obj))
print('Number of windows:', len(analysis_obj.windows))

## Curve Quantification

In [ ]:
from src.curve_quantification.options import get_quantification_funcs

quantification_funcs = get_quantification_funcs()
print('Available quantification functions:', quantification_funcs.keys())

In [ ]:
function_names = ['lognormal_fit_full']
output_path = '/Users/samantha/Desktop/tul/china data/p34/new_v1/manual_paramap/output.csv'
curve_quantifications_kwargs = {
    'curves_to_fit': ['TIC'],
    'tic_name': 'TIC'
}

In [ ]:
import os
import sys
from joblib import Parallel, delayed
from src.curve_quantification.functions import *
from src.entrypoints import curve_quantification_step


def _compute_single_window(analysis_obj, curves, function_names, kwargs):
    # Suppress stdout at OS level so prints from source code don't leak
    devnull_fd = os.open(os.devnull, os.O_WRONLY)
    old_fd = os.dup(1)
    os.dup2(devnull_fd, 1)
    os.close(devnull_fd)
    try:
        data_dict = {}
        data_dict['Scan Name'] = analysis_obj.image_data.scan_name
        data_dict['Segmentation Name'] = analysis_obj.seg_data.seg_name
        if 'Window-Axial Start Pix' in curves:
            data_dict['Window-Axial Start Pix'] = curves['Window-Axial Start Pix']
            data_dict['Window-Sagittal Start Pix'] = curves['Window-Sagittal Start Pix']
            data_dict['Window-Axial End Pix'] = curves['Window-Axial End Pix']
            data_dict['Window-Sagittal End Pix'] = curves['Window-Sagittal End Pix']
            if 'Window-Coronal Start Pix' in curves:
                data_dict['Window-Coronal Start Pix'] = curves['Window-Coronal Start Pix']
                data_dict['Window-Coronal End Pix'] = curves['Window-Coronal End Pix']
        for func_name in function_names:
            func = globals()[func_name]
            func(analysis_obj, curves, data_dict, **kwargs)
    finally:
        os.dup2(old_fd, 1)
        os.close(old_fd)
    return data_dict


curve_quant = curve_quantification_step(analysis_obj, function_names, output_path=None, **curve_quantifications_kwargs)

n_windows = len(analysis_obj.curves)
results = []
print_every = max(1, n_windows // 20)

for i, result in enumerate(Parallel(n_jobs=12, return_as='generator')(
    delayed(_compute_single_window)(analysis_obj, curves, curve_quant.ordered_func_names, curve_quantifications_kwargs)
    for curves in analysis_obj.curves
)):
    results.append(result)
    if (i + 1) % print_every == 0 or (i + 1) == n_windows:
        print(f'Quantifying curves: {i+1}/{n_windows} ({100*(i+1)/n_windows:.0f}%)')

# Fill missing keys across all windows
all_keys = set(k for d in results for k in d.keys())
for d in results:
    for k in all_keys:
        if k not in d:
            d[k] = None

curve_quant.data_dict = results

if output_path:
    import pandas as pd
    pd.DataFrame(results).to_csv(output_path, index=False)
    print(f'Saved to {output_path}')

print('curve_quant.analysis_objs type:', type(curve_quant.analysis_objs))
print('Windows quantified:', len(results))

## Parametric Map Saving

In [ ]:
from src.entrypoints import visualization_step

vis_type = 'paramap'
params = []
vis_funcs = []
vis_kwargs = {
    'paramap_folder_path': '/Users/samantha/Desktop/tul/china data/p34/new_v1/manual_paramap',
    'hide_all_visualizations': False,
}

vis_obj = visualization_step(curve_quant, vis_type, params, vis_funcs, **vis_kwargs)